# EuroSAT lab 3 · SHAP from local to global: band attributions for every image

**Author:** Xiangqi Kong

**Deck:** Post-hoc Model-Agnostic (*Shapley values*, *SHAP*, *local to global*) · **Follows:** the LIME lab · **Runtime:** about 5–10 minutes on a T4 with the lab pack · **GPU:** helps

PCI gives one number per band for the whole test set. LIME (lab 2) gives one band importance per image, but it is a regression fit with no
guarantees. **Shapley values** split one prediction fairly among the players, with properties LIME lacks:

$$\phi_j(x) = \sum_{S \subseteq N \setminus \{j\}} \frac{|S|!\,(n - |S| - 1)!}{n!}\,\big[v(S \cup \{j\}) - v(S)\big]$$

* **Players:** the $n = 14$ input channels (13 bands + the NOISE control).
* **Value function $v(S)$:** $p(\text{class})$ when only the channels in $S$ are kept and the others are set to their training mean.
* **Efficiency:** $\sum_j \phi_j = v(N) - v(\varnothing) = p(\text{class} \mid x) - p(\text{class} \mid \text{all bands at the mean})$.

With 14 players there are only $2^{14} = 16{,}384$ coalitions, so we can compute **exact** Shapley values for one image and check KernelSHAP
against them. Then we explain hundreds of images with KernelSHAP and aggregate: **beeswarm** (all local explanations at once), **mean |SHAP|**
(a global ranking to compare with PCI), per-class summaries, and grouped players for correlated bands.

**What you should see.** KernelSHAP matches the exact values closely with far fewer model calls. The global SHAP ranking broadly agrees with PCI
but not perfectly, because the two methods remove bands differently and measure different things. NOISE gets ≈ 0.

> **Easy version.** Almost all the code is written. Each **✏️ TODO** has exactly **one blank, written `BLANK`**, and the comment above it
> says what goes there. Replace `BLANK`, run the cell, then run the **✅ check** cell after it. If you see
> `NameError: name 'BLANK' is not defined`, a blank is still empty. The `_walkthrough` notebook in this folder explains every line; the `_solution` notebook has all the answers.

> **In the lab session: about 25 minutes.** Do the sections marked 🟢 core in order; they are what we discuss at the end.
> Sections marked ⚪ if time can be skipped: no 🟢 core section needs them, so come back to them at home.
>
> 🟢 core: 1 exact Shapley values (TODO 1, 2), 2 waterfall, 3 many images, 4 beeswarm, 5 mean |SHAP| vs PCI (TODO 3).  
> ⚪ if time: 6 per class (TODO 4), 7 dependence plot, 8 grouped players.

## 0. Setup

> **What this cell does.** It installs the few Python packages that Colab does not have. Just run it; it takes about 10 seconds.

In [ ]:
!pip -q install kagglehub tifffile shap

> **What this cell does.** It loads the libraries (NumPy, PyTorch, Matplotlib), fixes the random seed so everyone gets the same numbers, and checks whether a GPU is available. Just run it.

In [ ]:
import os, random, time, zipfile, urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
from tqdm.auto import tqdm

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = DEVICE.type == "cuda"
print("Device:", DEVICE, "| torch", torch.__version__)
if not GPU:
    print("No GPU found: the notebook still runs, with smaller sample sizes (see the settings below).")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})

### Settings
The data and model settings are copied from the PCI notebook. Change only the lab settings further down.

> **What this cell does.** It only sets names and options: the 10 land-cover classes, the 13 satellite bands and their spectral groups (with one colour per group for the plots), and `USE_LAB_PACK = True`, which means *download the ready-made model*. You do not need to change anything.

In [ ]:
# ---- data and model: identical to the PCI notebook, so every lab explains the same model on the same test set ----
DATA_DIR    = os.environ.get("EUROSAT_DIR", "")   # a folder of EuroSAT all-bands .tif files, to skip downloading
N_PER_CLASS = 1000
TEST_FRAC, VAL_FRAC = 0.20, 0.10
EPOCHS, BATCH_SIZE, LR = 12, 128, 3e-3
EVAL_BATCH  = 512

# ---- cache: the first lab you run trains the model and saves it with the test set; the other labs just load it ----
USE_LAB_PACK = True    # True: download the ready-made lab pack (the trained model + 500 test images, about 40 MB, a few seconds).
                       # False: download EuroSAT (2 GB) and train the model yourself (10-20 minutes on a T4), as in the PCI notebook.
LAB_PACK_URLS = ['https://github.com/kxqdesign/XAILabData/raw/main/lab-data/eurosat_lab_pack_v1.npz', 'https://github.com/kxqdesign/XAI-Course/raw/main/lab-session/lab-data/eurosat_lab_pack_v1.npz']
USE_DRIVE   = False     # True = keep the cache on Google Drive (Colab asks for permission once); useful with USE_LAB_PACK = False
CACHE_NAME  = "eurosat_smallcnn_seed42_labpack_v1.pt" if USE_LAB_PACK else "eurosat_smallcnn_seed42_v1.pt"

CLASSES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
           "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09", "B10", "B11", "B12", "B8A"]   # GeoTIFF order
BAND_INFO = {   # name: (description, resolution m, spectral group)
    "B01": ("Coastal aerosol", 60, "Atmospheric"), "B02": ("Blue", 10, "Visible"),
    "B03": ("Green", 10, "Visible"),               "B04": ("Red", 10, "Visible"),
    "B05": ("Red edge 1", 20, "Red edge"),         "B06": ("Red edge 2", 20, "Red edge"),
    "B07": ("Red edge 3", 20, "Red edge"),         "B08": ("NIR", 10, "NIR"),
    "B09": ("Water vapour", 60, "Atmospheric"),    "B10": ("SWIR cirrus", 60, "Atmospheric"),
    "B11": ("SWIR 1", 20, "SWIR"),                 "B12": ("SWIR 2", 20, "SWIR"),
    "B8A": ("Narrow NIR", 20, "NIR"),
}
GROUP_COLORS = {"Visible": "#2a78d6", "Red edge": "#eb6834", "NIR": "#1baf7a",
                "SWIR": "#eda100", "Atmospheric": "#e87ba4", "Control": "#8a8985"}
CHANNEL_NAMES = BANDS + ["NOISE"]          # channel 14 = the pure-noise control channel from the PCI notebook
N_CH = len(CHANNEL_NAMES)
GROUP_OF = {**{b: BAND_INFO[b][2] for b in BANDS}, "NOISE": "Control"}
GROUPS = {g: [i for i, c in enumerate(CHANNEL_NAMES) if GROUP_OF[c] == g] for g in GROUP_COLORS}

## 0. The model: load the lab pack (or train it yourself)

This is the **same** data pipeline and CNN as the PCI notebook (`Permutation_Channel_Importance_EuroSAT.ipynb`): the same 1,000 images per class,
the same seed and train/val/test split, per-band standardisation with training statistics, and the small CNN trained on the 13 bands plus a
pure-noise control channel. So the explanations in these labs are about the model whose channels PCI ranked.

**In the lab session you do not download EuroSAT or train anything.** With `USE_LAB_PACK = True` (the default) the next cell downloads a
small *lab pack* (about 40 MB, a few seconds): the model already trained with exactly this code on the real EuroSAT data, and 500 of its
test images (50 per class). Every lab in the session explains this same model.

To rebuild everything yourself after class, set `USE_LAB_PACK = False`: the cell then downloads EuroSAT all-bands (2 GB) and trains the
model (10–20 minutes on a T4), with the download code of the PCI notebook (Kaggle, then Hugging Face, then the DFKI server).

> **What this cell does.** **You can skip reading this cell; just run it.** It defines the model (a small CNN with four convolution blocks) and the functions that would download the full dataset. With the lab pack those download functions are never called.

In [ ]:
KAGGLE_SLUGS = ["apollo2506/eurosat-dataset"]
FALLBACK_URLS = [
    "https://huggingface.co/datasets/torchgeo/eurosat/resolve/main/EuroSATallBands.zip",
    "https://madm.dfki.de/files/sentinel/EuroSATallBands.zip",
]

def find_multiband_tifs(root):
    # Return {class_name: [paths]} for 13-band .tif files below root (deduplicated by filename).
    by_class, seen = defaultdict(list), set()
    for p in sorted(Path(root).rglob("*.tif")):
        cls = p.parent.name
        if cls in CLASSES and p.name not in seen:
            seen.add(p.name); by_class[cls].append(str(p))
    if not by_class:
        return {}
    probe = tifffile.imread(next(iter(by_class.values()))[0])
    if 13 not in probe.shape:
        print(f"  found .tif files in {root} but they have shape {probe.shape}, not 13 bands")
        return {}
    return dict(by_class)

def setup_kaggle_credentials():
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return
    try:                                   # Colab Secrets
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        print("  using Kaggle credentials from Colab Secrets")
    except Exception:
        pass

def try_kagglehub():
    import kagglehub
    for slug in KAGGLE_SLUGS:
        try:
            print(f"kagglehub: downloading {slug} ...")
            path = kagglehub.dataset_download(slug)
            found = find_multiband_tifs(path)
            if found: return found
        except Exception as e:
            print(f"  kagglehub failed for {slug}: {type(e).__name__}: {e}")
    return {}

def try_url_fallback():
    out = Path("/content/eurosat_ms" if Path("/content").exists() else "eurosat_ms")
    out.mkdir(exist_ok=True)
    for url in FALLBACK_URLS:
        zpath = out / "EuroSATallBands.zip"
        try:
            print(f"fallback: downloading {url} (~2 GB) ...")
            urllib.request.urlretrieve(url, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(out)
            zpath.unlink()
            found = find_multiband_tifs(out)
            if found: return found
        except Exception as e:
            print(f"  failed: {type(e).__name__}: {e}")
    return {}

def get_eurosat():
    if DATA_DIR:
        found = find_multiband_tifs(DATA_DIR)
        if found: return found
        print(f"DATA_DIR={DATA_DIR!r} has no 13-band EuroSAT tifs, trying downloads")
    setup_kaggle_credentials()
    for fn in (try_kagglehub, try_url_fallback):
        found = fn()
        if found:
            print(f"data source: {fn.__name__}")
            return found
    raise RuntimeError("Could not obtain EuroSAT all-bands data. Download it manually and set DATA_DIR.")

def read_tif(path):
    a = tifffile.imread(path)
    if a.ndim == 3 and a.shape[-1] == 13:        # (H, W, 13) -> (13, H, W)
        a = np.transpose(a, (2, 0, 1))
    return a

class SmallCNN(nn.Module):
    # The CNN from the PCI notebook: 4 conv blocks + global average pooling.
    def __init__(self, in_ch, n_classes=10, w=48):
        super().__init__()
        def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                              nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
        self.features = nn.Sequential(block(in_ch, w), nn.MaxPool2d(2), block(w, 2*w), nn.MaxPool2d(2),
                                      block(2*w, 4*w), nn.MaxPool2d(2), block(4*w, 4*w))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(4*w, n_classes))
    def forward(self, x): return self.head(self.features(x))

def augment(x):
    if random.random() < 0.5: x = x.flip(-1)
    if random.random() < 0.5: x = x.flip(-2)
    return torch.rot90(x, random.randint(0, 3), dims=(-2, -1))

> **What this cell does.** **Just run it.** It downloads the lab pack (about 40 MB, a few seconds), loads the trained model, and loads the 500 test images into `X`. Each image has 14 channels: 13 satellite bands plus one channel of pure noise that the model should ignore.

In [ ]:
def get_cache_path():
    folder = Path("xai_eurosat_cache")
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            folder = Path("/content/drive/MyDrive/xai_eurosat_cache")
        except Exception as e:
            print(f"Google Drive not available ({type(e).__name__}); caching in ./{folder}")
    folder.mkdir(parents=True, exist_ok=True)
    return folder / CACHE_NAME

def train_and_cache(path):
    # Same sampling, split, standardisation and training recipe as section E0 of the PCI notebook.
    files_by_class = get_eurosat()
    rng = np.random.default_rng(SEED)
    paths, labels = [], []
    for ci, cls in enumerate(CLASSES):
        ps = files_by_class.get(cls, []); take = min(N_PER_CLASS, len(ps))
        paths += list(rng.choice(ps, size=take, replace=False)); labels += [ci] * take
    X_raw = np.empty((len(paths), 13, 64, 64), dtype=np.uint16); keep = np.ones(len(paths), bool)
    for i, p in enumerate(tqdm(paths, desc="reading tifs")):
        a = read_tif(p)
        if a.shape != (13, 64, 64): keep[i] = False; continue
        X_raw[i] = a
    X_raw, y_all = X_raw[keep], np.array(labels)[keep]
    idx = np.arange(len(y_all))
    idx_trval, idx_test = train_test_split(idx, test_size=TEST_FRAC, stratify=y_all, random_state=SEED)
    idx_train, idx_val = train_test_split(idx_trval, test_size=VAL_FRAC / (1 - TEST_FRAC),
                                          stratify=y_all[idx_trval], random_state=SEED)
    mean = X_raw[idx_train].mean(axis=(0, 2, 3), dtype=np.float64)
    std = X_raw[idx_train].std(axis=(0, 2, 3), dtype=np.float64) + 1e-6
    def to_tensor(ix):
        x = (X_raw[ix].astype(np.float32) - mean[None, :, None, None].astype(np.float32)) / std[None, :, None, None].astype(np.float32)
        return torch.from_numpy(x).half()
    Xtr, Xva, Xte = (to_tensor(i) for i in (idx_train, idx_val, idx_test))
    ytr, yva, yte = (torch.from_numpy(y_all[i]).long() for i in (idx_train, idx_val, idx_test))
    del X_raw

    def with_noise(xb):                    # append a fresh noise channel, exactly as in the PCI notebook
        xb = xb.to(DEVICE).float()
        return torch.cat([xb, torch.randn_like(xb[:, :1])], 1)

    seed_everything()
    net = SmallCNN(N_CH).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * ((len(Xtr) + BATCH_SIZE - 1) // BATCH_SIZE))
    for ep in range(EPOCHS):
        net.train(); perm = torch.randperm(len(Xtr)); t0, tot = time.time(), 0.0
        for s in range(0, len(perm), BATCH_SIZE):
            b = perm[s:s + BATCH_SIZE]
            loss = F.cross_entropy(net(augment(with_noise(Xtr[b]))), ytr[b].to(DEVICE), label_smoothing=0.05)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * len(b)
        net.eval()
        with torch.no_grad():
            correct = sum((net(with_noise(Xva[s:s + EVAL_BATCH])).argmax(1).cpu() == yva[s:s + EVAL_BATCH]).sum().item()
                          for s in range(0, len(Xva), EVAL_BATCH))
        print(f"epoch {ep+1:2d}/{EPOCHS} loss {tot/len(perm):.3f} val_acc {correct/len(Xva):.3f} ({time.time()-t0:.0f}s)")
    torch.save(dict(state_dict={k: v.cpu() for k, v in net.state_dict().items()},
                    band_mean=torch.from_numpy(mean), band_std=torch.from_numpy(std),
                    X_test=Xte, y_test=yte, test_index=torch.from_numpy(idx_test)), path)
    print("saved", path)

def cache_from_lab_pack(path):
    # The lab pack holds the model trained by train_and_cache() on the real EuroSAT data (same recipe, seed and split) and a
    # class-balanced part of its test set (50 images per class, raw values). We rebuild the same cache file that train_and_cache() writes.
    local = path.with_suffix(".npz")
    for url in LAB_PACK_URLS:
        if local.exists(): break
        try:
            print(f"downloading the lab pack from {url} ..."); urllib.request.urlretrieve(url, local)
        except Exception as e:
            print(f"  failed: {type(e).__name__}: {e}"); local.unlink(missing_ok=True)
    if not local.exists():
        raise RuntimeError("Could not download the lab pack. Upload eurosat_lab_pack_v1.npz to the Colab files panel and run this cell again, "
                           "or set USE_LAB_PACK = False to download EuroSAT and train the model yourself.")
    z = np.load(local)
    mean, std = z["band_mean"], z["band_std"]
    x = (z["X_test_raw"].astype(np.float32) - mean[None, :, None, None].astype(np.float32)) / std[None, :, None, None].astype(np.float32)
    torch.save(dict(state_dict={k[3:]: torch.from_numpy(z[k]) for k in z.files if k.startswith("w::")},
                    band_mean=torch.from_numpy(mean), band_std=torch.from_numpy(std), X_test=torch.from_numpy(x).half(),
                    y_test=torch.from_numpy(z["y_test"]).long(), test_index=torch.from_numpy(z["test_index"])), path)
    print(f"lab pack ready: {len(x)} test images (the model scored {float(z['full_test_acc']):.4f} on the full 2,000-image test set)")

CACHE = get_cache_path()
UPLOADED = Path("eurosat_lab_pack_v1.npz")                      # a pack uploaded by hand to the Colab files panel, if the download fails
if USE_LAB_PACK and UPLOADED.exists() and not CACHE.with_suffix(".npz").exists():
    import shutil; shutil.copy(UPLOADED, CACHE.with_suffix(".npz"))
if not CACHE.exists():
    cache_from_lab_pack(CACHE) if USE_LAB_PACK else train_and_cache(CACHE)
ck = torch.load(CACHE, map_location="cpu", weights_only=True)
model = SmallCNN(N_CH).to(DEVICE); model.load_state_dict(ck["state_dict"]); model.eval()
band_mean, band_std = ck["band_mean"].numpy(), ck["band_std"].numpy()
y_test = ck["y_test"].numpy()

# Freeze the noise control channel: every test image gets one fixed noise image, so the model is a
# deterministic function of its 14-channel input and the explanation methods can treat NOISE like any other channel.
NOISE = torch.randn(len(ck["X_test"]), 1, 64, 64, generator=torch.Generator().manual_seed(SEED)).half()
X = torch.cat([ck["X_test"], NOISE], 1)       # (N, 14, 64, 64), standardised: 0 = the training mean of each band
del ck
print("test images:", tuple(X.shape))

> **What to notice.** The last line should read `test images: (500, 14, 64, 64)`: 500 images, 14 channels, 64 × 64 pixels.

### Prediction helpers and example images
`coalition_proba` is the one perturbation used by every method in these labs: *remove* a set of pixels or channels by setting them to the training mean, and read the probability of the class we explain.

> **What this cell does.** It defines four helpers used in the rest of the lab, then picks one example image per class.
> 
> - `predict_proba(x)`: the model's 10 class probabilities for an image.
> - `rgb(x)`: a true-colour picture of an image, for display only.
> - `show_map(...)`: draws an explanation on top of the picture (red = supports the class, blue = against it).
> - `coalition_proba(x, Z, target)`: **the key helper.** Each row of `Z` is a list of 1s and 0s: 1 = keep this part of the image, 0 = remove it (replace it by the average value). It returns the probability of the class `target` for every row. LIME and SHAP both work by calling this many times.

In [ ]:
@torch.no_grad()
def predict_proba(x, batch=EVAL_BATCH):
    # Class probabilities for a (N, 14, H, W) or (14, H, W) tensor, as a numpy array (N, 10).
    if x.ndim == 3: x = x[None]
    return torch.cat([F.softmax(model(x[s:s + batch].to(DEVICE).float()), 1).cpu()
                      for s in range(0, len(x), batch)]).numpy()

P = predict_proba(X); pred = P.argmax(1)
print(f"Test accuracy: {(pred == y_test).mean():.4f}   (the PCI notebook reports the same model's baseline)")

def rgb(x):
    # True-colour composite (B04, B03, B02) of a standardised image, for display only.
    x = x[:13].float().numpy() * band_std[:, None, None] + band_mean[:, None, None]
    img = np.stack([x[3], x[2], x[1]], -1)
    lo, hi = np.percentile(img, 2), np.percentile(img, 98)
    return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)

def show_map(ax, x, heat, title="", alpha=0.55, lim=None):
    # Overlay a signed attribution map on the true-colour image: red = supports the class, blue = against it.
    # `lim` fixes the colour scale (to compare panels); by default each panel uses its own maximum.
    ax.imshow(rgb(x)); lim = (np.abs(heat).max() if lim is None else lim) + 1e-8
    im = ax.imshow(heat, cmap="RdBu_r", vmin=-lim, vmax=lim, alpha=alpha)
    ax.set_title(title, fontsize=8); ax.axis("off"); ax.grid(False)
    return im

def coalition_proba(x, Z, target, segments=None, chunk=256):
    # p(target) when only the "players" switched on in each row of Z are kept and the rest are set to the
    # training mean (0 after standardisation). Players are superpixels if `segments` is given, else the 14 channels.
    x = x.float(); out = []
    seg = None if segments is None else torch.as_tensor(segments, dtype=torch.long)
    for s in range(0, len(Z), chunk):
        z = torch.as_tensor(np.asarray(Z[s:s + chunk]), dtype=torch.float32)
        m = z[:, seg][:, None] if seg is not None else z[:, :, None, None]
        out.append(predict_proba(x[None] * m)[:, target])
    return np.concatenate(out)

# Example images: per class, a correctly classified test image of MEDIAN confidence
# (very confident images saturate near p = 1, which hides the effect of a perturbation).
EXAMPLES = {}
for ci, cls in enumerate(CLASSES):
    ok = np.where((y_test == ci) & (pred == ci))[0]
    if len(ok) == 0: ok = np.where(y_test == ci)[0]
    ok = ok[np.argsort(P[ok, ci])]; EXAMPLES[cls] = int(ok[len(ok) // 2])
AMBIGUOUS = int(np.argmax(np.sort(P, 1)[:, -2]))     # the test image with the most probable runner-up class

fig, axes = plt.subplots(1, 11, figsize=(17, 2.2))
for ax, (cls, i) in zip(axes, list(EXAMPLES.items()) + [("ambiguous", AMBIGUOUS)]):
    top = np.argsort(P[i])[::-1][:2]
    ax.imshow(rgb(X[i])); ax.axis("off")
    title = (f"ambiguous\n{CLASSES[top[0]]} {P[i, top[0]]:.2f}\n{CLASSES[top[1]]} {P[i, top[1]]:.2f}" if cls == "ambiguous"
             else f"{cls}\np = {P[i, CLASSES.index(cls)]:.2f}")
    ax.set_title(title, fontsize=7)
plt.suptitle("Example test images used in this lab (true colour)"); plt.tight_layout(); plt.show()

> **What to notice.** The test accuracy is about 0.96, so the model is good. The pictures are the example images used below, with the model's confidence `p`. The last one is an image the model is unsure about.

### Lab settings

> **What this cell does.** Three settings: how many test images we explain, how many coalitions KernelSHAP samples per image, and whether to also compute the exact Shapley values for one image (slow on a CPU, a few seconds on a GPU).

In [ ]:
# ---- lab settings ----
N_EXPLAIN   = 300 if GPU else 60      # test images explained with KernelSHAP (stratified over classes)
NSAMPLES    = 1024                    # KernelSHAP coalitions per image (of 16,384 possible)
RUN_EXACT   = True                    # exact Shapley values for one image: 16,384 forward passes (~10 s on GPU, a few minutes on CPU)

## 1. Exact Shapley values for one image · 🟢 core

Enumerate all $2^{14}$ coalitions, evaluate $v(S)$ for each (one batched forward pass per coalition), and apply the formula above.

> **The idea in words.** The 14 channels are the *players*. A *coalition* is a subset of channels that are kept; all other channels are set to their average. The *value* of a coalition is the probability the model then gives to the class. A channel's Shapley value is its **average contribution**: over all coalitions without it, how much does the probability rise when it is added? Larger coalitions and smaller ones are weighted so that every order of adding the players counts the same.
> 
> **What this cell does.** `shapley_from_values` computes exact Shapley values from the values of all coalitions (your two blanks are here). `exact_shapley` evaluates all 2¹⁴ = 16,384 coalitions for one image. `kernel_shap` calls the `shap` package, which estimates the same numbers from only about 1,000 sampled coalitions.

In [ ]:
from math import factorial
import shap

def all_coalitions(n):
    m = np.arange(2 ** n)
    return m, (m[:, None] >> np.arange(n)) & 1                     # bitmask id, and the (2^n, n) on/off matrix

def shapley_from_values(v, n):
    # Shapley values of n players from the values v[S] of all 2^n coalitions (S = bitmask: bit j set = player j in S).
    ids = np.arange(2 ** n)
    size = np.array([bin(S).count("1") for S in ids])
    # ✏️ TODO 1 - the Shapley weight of a coalition of size s:   s! (n - s - 1)! / n!
    #   The first and the last factorial are written. Replace BLANK with the middle one's argument: n - s - 1
    wt = np.array([factorial(s) * factorial(BLANK) / factorial(n) for s in range(n)])
    phi = np.zeros(n)
    for j in range(n):
        S = ids[((ids >> j) & 1) == 0]                               # coalitions without player j (as bitmasks)
        # ✏️ TODO 2 - phi[j] = sum over these S of  weight * [ value WITH player j  -  value WITHOUT player j ].
        #   v[S | (1 << j)] is the value with player j added. Replace BLANK with the value without it: v[S]
        phi[j] = (wt[size[S]] * (v[S | (1 << j)] - BLANK)).sum()
    return phi

def exact_shapley(x, target, groups=None):
    # Exact Shapley values. Players = channels, or groups of channels (list of channel-index lists).
    groups = [[j] for j in range(N_CH)] if groups is None else groups
    n = len(groups); G = np.zeros((n, N_CH), int)
    for g, chans in enumerate(groups): G[g, chans] = 1
    ids, Zg = all_coalitions(n)
    v = coalition_proba(x, Zg @ G, target, chunk=EVAL_BATCH)       # value of every coalition
    return shapley_from_values(v, n), v[0], v[-1]                   # values, v(empty), v(all)

def kernel_shap(x, target, segments=None, nsamples=NSAMPLES, seed=0):
    # KernelSHAP from the shap package, with the same value function. Players = channels or superpixels.
    d = N_CH if segments is None else int(segments.max()) + 1
    np.random.seed(seed)
    ex = shap.KernelExplainer(lambda Z: coalition_proba(x, Z, target, segments), np.zeros((1, d)))
    sv = ex.shap_values(np.ones((1, d)), nsamples=nsamples, l1_reg=False, silent=True)
    return np.asarray(sv, dtype=float).reshape(-1)[:d], float(np.asarray(ex.expected_value).reshape(-1)[0])

### ✅ Check: `shapley_from_values` on games with known answers

In [ ]:
n = 3; ids = np.arange(2 ** n); size = np.array([bin(S).count("1") for S in ids])
a = np.array([0.5, -1.0, 2.0])
additive = np.array([sum(a[j] for j in range(n) if (S >> j) & 1) for S in ids])
assert np.allclose(shapley_from_values(additive, n), a), "additive game: each player's Shapley value is its own a_j"
assert np.allclose(shapley_from_values(size.astype(float) ** 2, n), [3, 3, 3]), "symmetric game v = |S|^2: equal shares of v(N) = 9"
glove = np.array([1.0 if ((S & 1) and (S & 6)) else 0.0 for S in ids])          # player 0 + (player 1 or 2)
assert np.allclose(shapley_from_values(glove, n), [2 / 3, 1 / 6, 1 / 6]), "glove game"
print("✅ Shapley values are right")

> **What this cell does.** It explains one River image twice: with KernelSHAP (1,024 sampled coalitions) and exactly (all 16,384 coalitions).

In [ ]:
i0 = EXAMPLES["River"]; c0 = CLASSES.index("River")
t = time.time(); ks, base = kernel_shap(X[i0], c0); t_ks = time.time() - t
print(f"KernelSHAP: {NSAMPLES} coalitions, {t_ks:.1f} s")
if RUN_EXACT:
    t = time.time(); ex_phi, v0, v1 = exact_shapley(X[i0], c0); t_ex = time.time() - t
    print(f"Exact:      {2**N_CH} coalitions, {t_ex:.1f} s")
    print(f"Efficiency: sum(phi) = {ex_phi.sum():.4f}   p(x) - p(all at mean) = {v1 - v0:.4f}")
    print(f"KernelSHAP vs exact: max |difference| = {np.abs(ks - ex_phi).max():.4f}, Spearman = {spearmanr(ks, ex_phi)[0]:.3f}")
    fig, ax = plt.subplots(figsize=(4.2, 4))
    ax.scatter(ex_phi, ks, c=[GROUP_COLORS[GROUP_OF[ch]] for ch in CHANNEL_NAMES], s=40)
    for j, ch in enumerate(CHANNEL_NAMES): ax.annotate(ch, (ex_phi[j], ks[j]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    lim = [min(ex_phi.min(), ks.min()), max(ex_phi.max(), ks.max())]; ax.plot(lim, lim, color="#8a8985", lw=1, ls="--")
    ax.set_xlabel("exact Shapley value"); ax.set_ylabel(f"KernelSHAP ({NSAMPLES} samples)"); ax.set_title("River example")
    plt.tight_layout(); plt.show()

> **What to notice.** `sum(phi)` equals `p(x) - p(all at mean)`: the Shapley values add up exactly to the change in the prediction (*efficiency*). In the plot the points lie on the diagonal: KernelSHAP gets almost the exact values with 16 times fewer model calls.

## 2. One prediction: the waterfall plot · 🟢 core

Start at $E[f] = v(\varnothing)$ (every band at its training mean), add one band's contribution at a time, end at the model's output for this
image. The grey numbers are the band's mean standardised value in this image (0 = an average pixel, positive = brighter than average).

> **What this cell does.** It draws the *waterfall plot* of the same River image.

In [ ]:
def band_features(x):
    return x.float().mean((-2, -1)).numpy()          # mean standardised value of every channel

phi = ex_phi if RUN_EXACT else ks
shap.plots.waterfall(shap.Explanation(values=phi, base_values=base, data=band_features(X[i0]).round(2),
                                      feature_names=CHANNEL_NAMES), max_display=N_CH)

> **What to notice.** Read it from the bottom: start at the prediction when every band is average, then each bar adds one band's Shapley value, and the top is the model's real output for this image. Red bars push the probability of River up, blue bars push it down.

## 3. Explain many images · 🟢 core

`N_EXPLAIN` test images, the same number per class. For each we explain $p(\text{true class})$ with KernelSHAP over the 14 channels.

> **What this cell does.** It runs KernelSHAP on many test images (the same number from each class) and stores the results in `SV`: one row per image, one column per channel. This takes about a minute on a GPU, a few minutes on a CPU.

In [ ]:
rng = np.random.default_rng(SEED)
per_class = max(1, N_EXPLAIN // len(CLASSES))
IDX = np.concatenate([rng.choice(np.where(y_test == ci)[0], size=min(per_class, (y_test == ci).sum()), replace=False) for ci in range(len(CLASSES))])
SV, BASE = np.zeros((len(IDX), N_CH)), np.zeros(len(IDX))
t = time.time()
for k, i in enumerate(tqdm(IDX, desc="KernelSHAP")):
    SV[k], BASE[k] = kernel_shap(X[i], int(y_test[i]))
FEAT = X[IDX].float().mean((-2, -1)).numpy()                       # per-image mean value of every channel
Y = y_test[IDX]
print(f"{len(IDX)} images, {time.time() - t:.0f} s")

## 4. From local to global: the beeswarm · 🟢 core

Each dot is **one image**: its position is that band's SHAP value for that image, its colour is how bright the band is in that image. Bands
are sorted by mean |SHAP|. Look for bands where colour and position line up (e.g. dark NIR → negative for the true class).

> **What this cell does.** It draws the *beeswarm plot*: all the local explanations at once.

In [ ]:
expl = shap.Explanation(values=SV, base_values=BASE, data=FEAT, feature_names=CHANNEL_NAMES)
shap.plots.beeswarm(expl, max_display=N_CH, show=False)
plt.title("KernelSHAP over 14 channels, one dot per image"); plt.tight_layout(); plt.show()

> **What to notice.** Each dot is one image. Its position is the band's SHAP value for that image; its colour says whether the band is bright (red) or dark (blue) in that image. Bands at the top have the widest spread, so they matter most. NOISE should be a thin line at 0.

## 5. Global ranking: mean |SHAP| vs PCI · 🟢 core

Average the absolute SHAP values over all explained images and put them next to PCI on the same model. PCI is recomputed here on the full test
set with the same code as the PCI notebook (pixel shuffle, $k = 3$) and with mean fill, the removal SHAP uses.

They answer different questions: PCI = *how much accuracy is lost* if a band's spatial information is destroyed; mean |SHAP| = *how much a
band moves the probability* of the true class, on average, with credit shared fairly among bands that work together.

> **What this cell does.** It turns the local explanations into a **global ranking** (your blank: the mean of |SHAP| per channel) and puts it next to PCI, the global method of the PCI lab, computed in two ways.

In [ ]:
def pci_scores(mode="shuffle", k=3):
    # PCI on the test set: accuracy drop when one channel is shuffled within each image (or set to its training mean).
    base_acc = (pred == y_test).mean(); out = {}
    for c, name in enumerate(tqdm(CHANNEL_NAMES, desc=f"PCI ({mode})")):
        accs = []
        for r in range(k if mode == "shuffle" else 1):
            g = torch.Generator().manual_seed(r); preds = []
            for s in range(0, len(X), EVAL_BATCH):
                xb = X[s:s + EVAL_BATCH].float().clone(); B, _, H, W = xb.shape
                if mode == "shuffle":
                    perm = torch.argsort(torch.rand(B, H * W, generator=g), 1)
                    xb[:, c] = torch.gather(xb[:, c].reshape(B, -1), 1, perm).reshape(B, H, W)
                else:
                    xb[:, c] = 0
                preds.append(predict_proba(xb).argmax(1))
            accs.append((np.concatenate(preds) == y_test).mean())
        out[name] = base_acc - np.mean(accs)
    return pd.Series(out)

# ✏️ TODO 3 - local to global: SV has one row per image and one column per channel (N x 14). We want one number per
#   channel: the mean of |SHAP| over the images, i.e. down the rows. Replace BLANK with the axis of the rows: 0
ranking = pd.DataFrame({"mean |SHAP|": np.abs(SV).mean(BLANK)}, index=CHANNEL_NAMES)
ranking["PCI (shuffle)"] = pci_scores("shuffle"); ranking["PCI (mean fill)"] = pci_scores("mean")
ranking = ranking.sort_values("mean |SHAP|", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col in zip(axes, ranking.columns):
    ax.bar(ranking.index, ranking[col], color=[GROUP_COLORS[GROUP_OF[ch]] for ch in ranking.index])
    ax.set_title(col); ax.tick_params(axis="x", rotation=90); ax.axhline(0, color="#52514e", lw=0.8)
axes[0].set_ylabel("Δ p(true class)"); axes[1].set_ylabel("accuracy drop"); axes[2].set_ylabel("accuracy drop")
plt.suptitle("Bands in mean |SHAP| order"); plt.tight_layout(); plt.show()
print("Spearman rank correlation between the global rankings:")
display(ranking.corr(method="spearman").round(2))

> **What to notice.** The three bar charts mostly agree on which bands matter and all put NOISE near 0, but the order is not identical. The table gives the rank correlation between the three rankings.

## 6. Global per class · ⚪ if time

Average the **signed** SHAP values per true class: which bands push the probability of each class up (red) or down (blue)? Compare with the
per-class PCI heatmap (E2) in the PCI notebook.

> **What this cell does.** It averages the SHAP values separately for the images of each class.

In [ ]:
from matplotlib.colors import TwoSlopeNorm
# ✏️ TODO 4 - one row per class: the mean SHAP value of every channel over the images of that class only.
#   SV[Y == k] keeps the images whose true class is k. The loop variable is ci. Replace BLANK with it: ci
per_cls = pd.DataFrame([SV[Y == BLANK].mean(0) for ci in range(len(CLASSES))], index=CLASSES, columns=CHANNEL_NAMES)[ranking.index]
lim = np.abs(per_cls.values).max()
fig, ax = plt.subplots(figsize=(11, 4.8))
im = ax.imshow(per_cls.values, cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim), aspect="auto"); ax.grid(False)
ax.set_xticks(range(N_CH), per_cls.columns); ax.set_yticks(range(len(CLASSES)), CLASSES)
for r in range(len(CLASSES)):
    for k in range(N_CH):
        v = per_cls.values[r, k]
        if abs(v) >= 0.02: ax.text(k, r, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if abs(v) > .6 * lim else "#0b0b0b")
plt.colorbar(im, label="mean SHAP value"); ax.set_title("Mean SHAP value per true class")
plt.tight_layout(); plt.show()

> **What to notice.** Each row is a class. Red cells are bands that raise the probability of that class, blue cells lower it. Different classes rely on different bands.

## 7. Dependence plot · ⚪ if time

For the band with the largest mean |SHAP|: its SHAP value against its brightness in the image, coloured by class. A trend shows *how* the model
uses the band; different colours following different trends show interactions with the class (and with other bands).

> **What this cell does.** For the most important band, it plots the band's SHAP value against how bright the band is in each image.

In [ ]:
b = CHANNEL_NAMES.index(ranking.index[0])
fig, ax = plt.subplots(figsize=(7.5, 4))
cmap = plt.get_cmap("tab10")
for ci, cls in enumerate(CLASSES):
    m = Y == ci; ax.scatter(FEAT[m, b], SV[m, b], s=14, color=cmap(ci), label=cls, alpha=0.8)
ax.axhline(0, color="#52514e", lw=0.8)
ax.set_xlabel(f"mean standardised value of {CHANNEL_NAMES[b]} ({BAND_INFO[CHANNEL_NAMES[b]][0]}) in the image")
ax.set_ylabel("SHAP value (Δ p(true class))"); ax.legend(frameon=False, fontsize=7, ncol=2, bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

> **What to notice.** A clear trend shows *how* the model uses the band, for example: the darker the band, the more it supports water classes.

## 8. Correlated bands: grouped players · ⚪ if time

B08 and B8A both measure near-infrared and are almost perfectly correlated. When either one is present, the model has its NIR information, so
the Shapley value **splits** the credit between them. PCI showed the same effect (E4: grouped PCI). Treating spectral groups as **one player
each** (6 players, $2^6 = 64$ coalitions: exact and cheap) gives the group's credit directly.

Compare the group Shapley value with the sum of its bands' individual Shapley values.

> **What this cell does.** Bands of one spectral group are strongly correlated, so they share the credit. Here each group plays as **one** player and we compare its Shapley value with the sum over its single bands.

In [ ]:
gnames = [g for g in GROUPS if GROUPS[g]]
GSV = np.array([exact_shapley(X[i], int(y_test[i]), groups=[GROUPS[g] for g in gnames])[0] for i in tqdm(IDX, desc="group Shapley")])
cmp = pd.DataFrame({"group as one player": np.abs(GSV).mean(0),
                    "sum of its bands (|sum| per image)": [np.abs(SV[:, GROUPS[g]].sum(1)).mean() for g in gnames]}, index=gnames)
ax = cmp.plot.bar(figsize=(7.5, 3.6), color=["#2a78d6", "#86b6ef"], rot=0)
ax.set_ylabel("mean |Shapley value|"); ax.set_title("Spectral groups: grouped vs individual players"); plt.tight_layout(); plt.show()
display(cmp.round(4))

> **What to notice.** Where the two bars differ, single-band values are misleading for that group: correlated bands should be reported together.

## 9. Questions for discussion

1. Section 1: KernelSHAP used 1,024 coalitions instead of 16,384. How close was it? What would you do with 50 bands, or with 40 superpixels?
2. Check the efficiency line. Which baseline does every SHAP value here start from, and why does the choice of "training mean" matter?
3. In the beeswarm, which band shows the clearest relation between colour (brightness) and SHAP value? Is it physically plausible?
4. Mean |SHAP| and PCI rank the bands differently in places. Give two reasons from the definitions (metric, removal, interactions).
5. Did NOISE get ≈ 0 everywhere? Does the beeswarm show a narrow line at 0 for NOISE?
6. In section 8, is the NIR group worth more than the sum of B08 and B8A? What does that mean for reporting single-band importances?

**Try it.**
* Explain $p(\text{predicted class})$ for the misclassified images only. Which bands pushed the model to the wrong class?
* Replace the probability by the log-odds (`torch.log(p / (1 - p))`) in the value function. Does the ranking change?
* Set `NSAMPLES = 64` and repeat section 1. How far is KernelSHAP from the exact values now?

**References:** S. M. Lundberg and S.-I. Lee, *A Unified Approach to Interpreting Model Predictions*, NeurIPS 2017. L. S. Shapley, *A value for
n-person games*, 1953.